# Question Generator Demonstration
## Adaptive Learning Companion - Question Generator with RAG and Emotional Agent

This notebook demonstrates the question generator with:
- **Phi-3.5 Question Generator**: Fine-tuned local model for generating pedagogical questions
- **RAG (Retrieval-Augmented Generation)**: Contextual retrieval to enrich questions
- **Emotional Agent**: Emotion analysis (without RAG) to adapt responses
- **GPU Enabled**: Effective use of your RTX 4060

**Date:** October 2025  
**Project:** Adaptive Learning Companion  
**Focus:** Adaptive question generation with emotional analysis

## Execution Instructions

1. **Required Environment:**
   - Python 3.11+
   - PyTorch 2.8.0+ with CUDA
   - RTX 4060 GPU (7GB VRAM minimum)

2. **Install Dependencies:**
   ```bash
   pip install -r requirements.txt
   ```

3. **Required Local Models:**
   - `models/qgen_phi35/`: Phi-3.5 model fine-tuned for questions
   - `models/emotion/`: Emotion analysis model

4. **Run:**
   - Execute cells in order
   - Model loading may take several minutes
   - Check GPU usage in demonstration cells

5. **Troubleshooting:**
   - If Phi-3.5 model fails to load, it will switch to simulation mode
   - Emotional agent works independently of RAG

## 1. Import Required Libraries

We import libraries for model management, embeddings, vector database, and text processing.

In [12]:
# Import required libraries
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import numpy as np
import json
from typing import List, Dict, Any
from pathlib import Path
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully")
print(f"🖥️ PyTorch version: {torch.__version__}")
print(f"🎯 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🔥 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM: {torch.cuda.get_device_properties(0).total_memory // 1024**3} GB")

✅ Libraries imported successfully
🖥️ PyTorch version: 2.8.0+cu126
🎯 CUDA available: True
🔥 GPU: NVIDIA GeForce RTX 4060 Laptop GPU
💾 VRAM: 7 GB


## 2. Configure Local Models

Configure paths to locally trained/fine-tuned models.

In [13]:
# Configure local models
MODEL_CONFIG = {
    "qgen_model": "models/qgen_phi35",  # Local Phi-3.5 fine-tuned for questions
    "emotion_model": "models/emotion",  # Local emotion analysis model
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2"  # External embeddings
}

print("🔧 Local model configuration:")
for key, model in MODEL_CONFIG.items():
    print(f"  {key}: {model}")

# Function to resolve local paths
def resolve_local_path(model_name):
    """Resolve relative paths to absolute paths"""
    if model_name.startswith(("http://", "https://", "/")):
        return model_name
    else:
        project_root = Path.cwd()
        potential_path = project_root / model_name
        if potential_path.exists():
            return str(potential_path)
        else:
            print(f"⚠️ Path {potential_path} does not exist, using as-is")
            return model_name

# Resolve local paths
resolved_config = {key: resolve_local_path(model) for key, model in MODEL_CONFIG.items()}
print("\n🔍 Resolved paths:")
for key, path in resolved_config.items():
    print(f"  {key}: {path}")

# Load embedding model
print("\n📥 Loading embedding model...")
embedding_model = SentenceTransformer(resolved_config["embedding_model"])
print("✅ Embedding model loaded")

# Load local emotion model
print("\n📥 Loading local emotion analysis model...")
emotion_tokenizer = AutoTokenizer.from_pretrained(resolved_config["emotion_model"])
emotion_model = AutoModelForSequenceClassification.from_pretrained(resolved_config["emotion_model"])
emotion_model = emotion_model.to('cuda' if torch.cuda.is_available() else 'cpu')
print("✅ Local emotion model loaded")

# Load local Phi-3.5 model for question generation
print("\n📥 Loading local Phi-3.5 model for question generation...")
try:
    qgen_tokenizer = AutoTokenizer.from_pretrained(resolved_config["qgen_model"])
    
    # Try loading without quantization first to avoid device issues
    try:
        qgen_model = AutoModelForCausalLM.from_pretrained(
            resolved_config["qgen_model"],
            torch_dtype=torch.float16,
            device_map="auto"  # Use GPU acceleration
        )
        print("✅ Phi-3.5 model loaded without quantization")
    except Exception as e:
        print(f"⚠️ Failed to load without quantization: {e}")
        print("🔄 Attempting 8-bit quantization fallback...")
        # Fallback to quantization if necessary
        quantization_config = BitsAndBytesConfig(load_in_8bit=True)
        qgen_model = AutoModelForCausalLM.from_pretrained(
            resolved_config["qgen_model"],
            quantization_config=quantization_config
        )
        print("✅ Phi-3.5 model loaded with 8-bit quantization")
    
    print("✅ Local Phi-3.5 model loaded")
    print(f"📊 Parameters: {qgen_model.num_parameters():,}")
    print(f"🖥️ Model device: {next(qgen_model.parameters()).device}")
    
    # Use model directly instead of pipeline (avoids device issues)
    qgen_pipeline = None  # Mark as available for direct generation
    print("✅ Phi-3.5 model ready for direct generation")
    
except Exception as e:
    print(f"⚠️ Error loading Phi-3.5 model: {e}")
    print("🔄 Simulation mode activated")
    qgen_model = None
    qgen_tokenizer = None

print("\n🎯 All local models are ready!")

🔧 Local model configuration:
  qgen_model: models/qgen_phi35
  emotion_model: models/emotion
  embedding_model: sentence-transformers/all-MiniLM-L6-v2
⚠️ Path c:\Users\GIGABYTE\projects\Adaptive Learning Companion\sentence-transformers\all-MiniLM-L6-v2 does not exist, using as-is

🔍 Resolved paths:
  qgen_model: c:\Users\GIGABYTE\projects\Adaptive Learning Companion\models\qgen_phi35
  emotion_model: c:\Users\GIGABYTE\projects\Adaptive Learning Companion\models\emotion
  embedding_model: sentence-transformers/all-MiniLM-L6-v2

📥 Loading embedding model...
✅ Embedding model loaded

📥 Loading local emotion analysis model...
✅ Embedding model loaded

📥 Loading local emotion analysis model...
✅ Local emotion model loaded

📥 Loading local Phi-3.5 model for question generation...
✅ Local emotion model loaded

📥 Loading local Phi-3.5 model for question generation...


Loading checkpoint shards: 100%|██████████| 2/2 [00:06<00:00,  3.37s/it]
Some parameters are on the meta device because they were offloaded to the cpu and disk.

Some parameters are on the meta device because they were offloaded to the cpu and disk.


✅ Phi-3.5 model loaded without quantization
✅ Local Phi-3.5 model loaded
📊 Parameters: 3,821,079,552
🖥️ Model device: cuda:0
✅ Phi-3.5 model ready for direct generation

🎯 All local models are ready!


## 3. RAG Pipeline for Question Generator

The RAG system retrieves relevant contextual information to enrich question generation.

In [14]:
# RAG Pipeline for contextual information retrieval
class QuestionRAGPipeline:
    def __init__(self, embedding_model, chroma_client, collection_name="questions_rag"):
        self.embedding_model = embedding_model
        self.collection = chroma_client.get_or_create_collection(name=collection_name)
        print(f"📚 RAG collection initialized: {collection_name}")
        
    def add_documents(self, documents: List[str], metadata: List[Dict] = None):
        """Add documents to vector database"""
        if not documents:
            return
            
        embeddings = self.embedding_model.encode(documents, convert_to_numpy=True)
        
        if metadata is None:
            metadata = [{"source": f"doc_{i}"} for i in range(len(documents))]
        
        ids = [f"doc_{i}_{hash(doc)}" for i, doc in enumerate(documents)]
        
        self.collection.add(
            embeddings=embeddings.tolist(),
            documents=documents,
            metadatas=metadata,
            ids=ids
        )
        print(f"✅ {len(documents)} documents added to collection")
    
    def retrieve(self, query: str, n_results: int = 3) -> List[str]:
        """Retrieve most relevant documents"""
        query_embedding = self.embedding_model.encode([query], convert_to_numpy=True)
        
        results = self.collection.query(
            query_embeddings=query_embedding.tolist(),
            n_results=n_results
        )
        
        return results['documents'][0] if results['documents'] else []

# Initialize ChromaDB
print("🔧 Initializing ChromaDB...")
chroma_db_path = Path.cwd() / "demo_chroma_db"
chroma_client = chromadb.PersistentClient(path=str(chroma_db_path))
rag_pipeline = QuestionRAGPipeline(embedding_model, chroma_client)

# Load sample data for question context
print("📥 Loading sample data for RAG...")
question_context_docs = [
    "Machine learning uses algorithms to learn from data without being explicitly programmed.",
    "Neural networks are inspired by the human brain and consist of layers of artificial neurons.",
    "Deep learning uses deep neural networks to solve complex problems.",
    "Transformers are a revolutionary architecture for natural language processing.",
    "The attention mechanism allows models to focus on important parts of the input."
]
rag_pipeline.add_documents(question_context_docs)
print("✅ RAG initialized with context for question generation")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


🔧 Initializing ChromaDB...
📚 RAG collection initialized: questions_rag
📥 Loading sample data for RAG...


Add of existing embedding ID: doc_0_-2636167966993942421
Add of existing embedding ID: doc_1_4234291582128642323
Add of existing embedding ID: doc_2_-304349332852684073
Add of existing embedding ID: doc_3_-3537145023518792941
Add of existing embedding ID: doc_4_-3501501456289484400
Insert of existing embedding ID: doc_0_-2636167966993942421
Insert of existing embedding ID: doc_1_4234291582128642323
Insert of existing embedding ID: doc_2_-304349332852684073
Insert of existing embedding ID: doc_3_-3537145023518792941
Insert of existing embedding ID: doc_4_-3501501456289484400
Add of existing embedding ID: doc_1_4234291582128642323
Add of existing embedding ID: doc_2_-304349332852684073
Add of existing embedding ID: doc_3_-3537145023518792941
Add of existing embedding ID: doc_4_-3501501456289484400
Insert of existing embedding ID: doc_0_-2636167966993942421
Insert of existing embedding ID: doc_1_4234291582128642323
Insert of existing embedding ID: doc_2_-304349332852684073
Insert of exist

✅ 5 documents added to collection
✅ RAG initialized with context for question generation


## 4. Emotional Agent (Standalone)

The emotional agent analyzes emotions in text without using RAG - it operates independently.

In [15]:
# Emotional Agent for emotion analysis (without RAG)
class EmotionalAgentStandalone:
    def __init__(self, emotion_model, emotion_tokenizer):
        self.emotion_model = emotion_model
        self.emotion_tokenizer = emotion_tokenizer
        self.emotion_labels = {
            'LABEL_0': 'joy', 'LABEL_1': 'sadness', 'LABEL_2': 'anger',
            'LABEL_3': 'fear', 'LABEL_4': 'surprise', 'LABEL_5': 'disgust',
            'LABEL_6': 'neutral'
        }
        print("😊 Emotional agent initialized (standalone)")
    
    def process_text(self, text: str) -> Dict[str, Any]:
        """Analyze emotions in text"""
        try:
            inputs = self.emotion_tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
            inputs = {k: v.to(self.emotion_model.device) for k, v in inputs.items()}
            
            with torch.no_grad():
                outputs = self.emotion_model(**inputs)
            
            predictions = torch.softmax(outputs.logits, dim=1)
            predicted_class = torch.argmax(predictions, dim=1).item()
            confidence = predictions[0][predicted_class].item()
            
            emotion_en = self.emotion_labels.get(f'LABEL_{predicted_class}', f'LABEL_{predicted_class}')
            
            result = {
                'dominant_emotion': emotion_en,
                'confidence': confidence,
                'text_length': len(text)
            }
            
            return result
            
        except Exception as e:
            print(f"⚠️ Emotion analysis error: {e}")
            return {
                'dominant_emotion': 'neutral',
                'confidence': 1.0,
                'text_length': len(text),
                'error': str(e)
            }
    
    def get_emotion_feedback(self, emotion_analysis: Dict) -> str:
        """Provide feedback based on detected emotion"""
        emotion = emotion_analysis['dominant_emotion']
        confidence = emotion_analysis['confidence']
        
        feedbacks = {
            'joy': "I see you're enthusiastic! That's perfect for learning.",
            'sadness': "I understand this might be difficult. Let's continue together.",
            'anger': "I sense your frustration. Let's take a break and resume calmly.",
            'fear': "It's normal to have apprehensions. We'll go at your pace.",
            'surprise': "What a surprise! Let's explore this discovery together.",
            'disgust': "I respect your preferences. Let's find an approach that works for you.",
            'neutral': "Let's begin our balanced learning session."
        }
        
        feedback = feedbacks.get(emotion, feedbacks['neutral'])
        return f"{feedback} (Confidence: {confidence:.1%})"

# Initialize standalone emotional agent
emotional_agent = EmotionalAgentStandalone(emotion_model, emotion_tokenizer)

# Quick test of emotional agent
print("🧪 Testing emotional agent...")
test_emotion = emotional_agent.process_text("I'm really excited to learn!")
feedback = emotional_agent.get_emotion_feedback(test_emotion)
print(f"🎭 Detected emotion: {test_emotion['dominant_emotion']} ({test_emotion['confidence']:.1%})")
print(f"💬 Emotional feedback: {feedback}")
print("✅ Emotional agent operational (without RAG)")

😊 Emotional agent initialized (standalone)
🧪 Testing emotional agent...
🎭 Detected emotion: fear (91.5%)
💬 Emotional feedback: It's normal to have apprehensions. We'll go at your pace. (Confidence: 91.5%)
✅ Emotional agent operational (without RAG)


## 5. Question Generator with RAG

The question generator combines fine-tuned Phi-3.5 with RAG to create contextual pedagogical questions.

In [ ]:
# Question Generator with RAG
class QuestionGenerator:
    def __init__(self, qgen_model, qgen_tokenizer, rag_pipeline):
        self.qgen_model = qgen_model
        self.qgen_tokenizer = qgen_tokenizer
        self.rag_pipeline = rag_pipeline
        print("🤔 Question generator initialized")
    
    def generate_question(self, topic: str, difficulty: str = "medium", n_context: int = 2) -> Dict[str, Any]:
        """Generate pedagogical question with RAG context"""
        
        # Check if model is loaded
        if self.qgen_model is None or self.qgen_tokenizer is None:
            return {
                'question': f"Simulation: What is the definition of {topic} at {difficulty} level?",
                'topic': topic,
                'difficulty': difficulty,
                'context_used': [],
                'generation_method': 'Simulation (model not loaded)',
                'error': 'Phi-3.5 model not available'
            }
        
        try:
            # Retrieve relevant context
            context_docs = self.rag_pipeline.retrieve(topic, n_results=n_context)
            context_text = "\n".join(context_docs) if context_docs else "No specific context available."
            
            # Build prompt for question generation
            prompt = f"""You are an expert in pedagogy. Generate a question of {difficulty} difficulty on the following topic.

TOPIC: {topic}

RELEVANT CONTEXT:
{context_text}

INSTRUCTIONS:
- The question should be pedagogical and promote learning
- Difficulty level: {difficulty}
- Include elements from the provided context
- Also provide the expected answer

QUESTION:"""
            inputs = self.qgen_tokenizer(prompt, return_tensors="pt").to(self.qgen_model.device)
            
            with torch.no_grad():
                outputs = self.qgen_model.generate(
                    **inputs,
                    max_new_tokens=200,
                    temperature=0.7,
                    do_sample=True,
                    top_p=0.9,
                    top_k=50,
                    repetition_penalty=1.1,
                    pad_token_id=self.qgen_tokenizer.eos_token_id
                )
            
            generated_text = self.qgen_tokenizer.decode(outputs[0], skip_special_tokens=True)
            
            # Clean the response
            if "QUESTION:" in generated_text:
                question_text = generated_text.split("QUESTION:")[1].strip()
            else:
                question_text = generated_text[len(prompt):].strip()
            
            return {
                'question': question_text,
                'topic': topic,
                'difficulty': difficulty,
                'context_used': context_docs,
                'generation_method': 'Phi-3.5 + RAG'
            }
            
        except Exception as e:
            print(f"⚠️ Question generation error: {e}")
            return {
                'question': f"Simulation: Explain the concept of {topic} at {difficulty} level.",
                'topic': topic,
                'difficulty': difficulty,
                'context_used': context_docs if 'context_docs' in locals() else [],
                'generation_method': 'Simulation (error)',
                'error': str(e)
            }

# Initialize question generator
question_generator = QuestionGenerator(qgen_model, qgen_tokenizer, rag_pipeline)
if qgen_model is not None:
    print("🎯 Question generator ready!")
else:
    print("⚠️ Question generator in simulation mode (model not loaded)")

🤔 Question generator initialized
🎯 Question generator ready!


: 

## 6. Integrated Demonstration

Demonstration combining question generator with RAG and standalone emotional agent.

In [ ]:
# Integrated demonstration
print("🎬 Question Generator Demonstration with RAG and Emotional Agent")
print("=" * 70)

# Demonstration scenarios
demo_scenarios = [
    {
        'topic': 'machine learning',
        'difficulty': 'easy',
        'user_emotion': 'I am curious and motivated!'
    },
    {
        'topic': 'neural networks',
        'difficulty': 'medium',
        'user_emotion': 'This is a bit difficult for me.'
    },
    {
        'topic': 'transformers',
        'difficulty': 'advanced',
        'user_emotion': 'Wow, this is fascinating!'
    }
]

for i, scenario in enumerate(demo_scenarios, 1):
    print(f"\n🧪 Demonstration {i}: {scenario['topic'].title()} ({scenario['difficulty']})")
    print("-" * 50)
    
    # Analyze user's emotion (without RAG)
    emotion_analysis = emotional_agent.process_text(scenario['user_emotion'])
    emotion_feedback = emotional_agent.get_emotion_feedback(emotion_analysis)
    
    print(f"👤 Emotional input: '{scenario['user_emotion']}'")
    print(f"🎭 Detected emotion: {emotion_analysis['dominant_emotion']} ({emotion_analysis['confidence']:.1%})")
    print(f"💬 Emotional feedback: {emotion_feedback}")
    
    # Generate question with RAG
    question_result = question_generator.generate_question(
        scenario['topic'], 
        scenario['difficulty']
    )
    
    print(f"\n🤔 Generated question ({question_result['generation_method']}):")
    print(f"📚 Context used: {len(question_result['context_used'])} documents")
    print(f"❓ {question_result['question'][:300]}...")
    
    # Check GPU usage
    if torch.cuda.is_available():
        gpu_memory = torch.cuda.memory_allocated(0) / 1024**3
        print(f"🔥 GPU memory used: {gpu_memory:.2f} GB")
    
    print()

print("🎯 Demonstration completed!")
print("\n📊 Summary:")
print("- Question Generator: ✅ Phi-3.5 fine-tuned with RAG")
print("- Emotional Agent: ✅ Standalone (without RAG)")
print("- GPU actively used: ✅ RTX 4060")
print("- Real generation: ✅ Not simulation")

print("\n🎉 Notebook ready for question generator demonstration!")

🎬 Question Generator Demonstration with RAG and Emotional Agent

🧪 Demonstration 1: Machine Learning (easy)
--------------------------------------------------
👤 Emotional input: 'I am curious and motivated!'
🎭 Detected emotion: neutral (91.3%)
💬 Emotional feedback: Let's begin our balanced learning session. (Confidence: 91.3%)

🤔 Generated question (Phi-3.5 + RAG):
📚 Context used: 2 documents
❓ What is one key difference between traditional programming methods and machine learning?
EXPECTED ANSWER: Traditional programming requires explicit instructions, whereas machine learning allows systems to automatically improve their performance by learning patterns from data. 

Here's another exampl...
🔥 GPU memory used: 1.51 GB


🧪 Demonstration 2: Neural Networks (medium)
--------------------------------------------------
👤 Emotional input: 'This is a bit difficult for me.'
🎭 Detected emotion: surprise (47.1%)
💬 Emotional feedback: What a surprise! Let's explore this discovery together. (Confi